# Stammbaum-Tutorial mit UnifyWeaver

Dieses interaktive Notebook demonstriert, wie Sie UnifyWeaver verwenden, um Prolog-Prädikate in Bash-Skripte zu kompilieren.

## Voraussetzungen

- SWI-Prolog installiert
- UnifyWeaver-Bibliothek verfügbar
- Prolog-Jupyter-Kernel installiert (`pip install prolog-jupyter-kernel`)

## Lernziele

Am Ende dieses Notebooks werden Sie in der Lage sein:
1. Prolog-Fakten und -Regeln zu definieren
2. UnifyWeaver zum Kompilieren von Prädikaten nach Bash zu verwenden
3. Die generierten Bash-Skripte zu testen
4. Die Kompilierung transitiver Hüllen zu verstehen

## Schritt 1: UnifyWeaver-Umgebung initialisieren

Zuerst müssen wir die UnifyWeaver-Module laden. Wir verwenden die Datei `init.pl` aus dem Education-Verzeichnis.

In [ ]:
% Load the initialization file
['../init'].

## Schritt 2: Familienbeziehungen definieren

Definieren wir einige Eltern-Kind-Beziehungen aus dem biblischen Stammbaum.

In [ ]:
% Define parent facts
:- dynamic parent/2.

parent(abraham, isaac).
parent(abraham, ishmael).
parent(isaac, jacob).
parent(isaac, esau).
parent(jacob, reuben).
parent(jacob, simeon).
parent(jacob, levi).
parent(jacob, judah).

## Schritt 3: Eltern-Abfragen testen

Vor dem Kompilieren überprüfen wir mit einigen Prolog-Abfragen, ob unsere Daten korrekt sind.

In [ ]:
% Query: Who are Abraham's children?
parent(abraham, Child).

In [ ]:
% Query: Who are Jacob's children?
parent(jacob, Child).

## Schritt 4: Vorfahren-Beziehung definieren

Nun definieren wir die transitive Hülle — die `ancestor`-Relation.

In [ ]:
% Define ancestor as transitive closure of parent
:- dynamic ancestor/2.

% Base case: parent is an ancestor
ancestor(X, Y) :- parent(X, Y).

% Recursive case: if X is parent of Y and Y is ancestor of Z, then X is ancestor of Z
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).

## Schritt 5: Vorfahren-Abfragen testen

Überprüfen wir, ob unser Vorfahren-Prädikat korrekt funktioniert.

In [ ]:
% Query: Is Abraham an ancestor of Jacob?
( ancestor(abraham, jacob) ->
    writeln('Yes: Abraham is an ancestor of Jacob')
;
    writeln('No: Abraham is not an ancestor of Jacob')
).

In [ ]:
% Query: Who are all of Abraham's descendants?
ancestor(abraham, Descendant).

## Schritt 6: Parent nach Bash kompilieren

Nun zum spannenden Teil — kompilieren wir unsere `parent/2`-Fakten in ein Bash-Skript!

In [ ]:
% Load the stream compiler
\+ \+ (
    use_module(unifyweaver(core/stream_compiler)),

    % Compile parent facts to bash
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    writeln('Generated Bash code for parent/2:'),
    writeln(_BashCode)
).

## Schritt 7: Parent-Skript speichern

Speichern wir den generierten Bash-Code in einer Datei.

In [ ]:
% Save to file
\+ \+ (
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    setup_call_cleanup(
        open('../output/parent.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/parent.sh')
).

## Schritt 8: Ancestor nach Bash kompilieren

Nun kompilieren wir das Prädikat `ancestor/2`, das Rekursion verwendet.

In [ ]:
% Load the recursive compiler
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),

    % Compile ancestor to bash
    compile_recursive(ancestor/2, [], _BashCode),
    writeln('Generated Bash code for ancestor/2:'),
    writeln(_BashCode)
).

## Schritt 9: Ancestor-Skript speichern

Speichern Sie das Ancestor-Skript in einer Datei.

In [ ]:
% Save to file
\+ \+ (
    compile_recursive(ancestor/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/ancestor.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/ancestor.sh')
).

## Schritt 10: Die generierten Skripte testen

Testen wir nun unsere generierten Bash-Skripte! Wir verwenden das `%%bash`-Magic-Kommando, um Bash-Befehle auszuführen.

In [ ]:
%%bash
# Source the parent script
source ../output/parent.sh

# Test: Who are Abraham's children?
echo "Abraham's children:"
parent abraham

In [ ]:
%%bash
# Source both scripts
source ../output/parent.sh
source ../output/ancestor.sh

# Test: Who are Abraham's descendants?
echo "Abraham's descendants:"
ancestor abraham

In [ ]:
%%bash
# Source both scripts
source ../output/parent.sh
source ../output/ancestor.sh

# Test: Is Abraham an ancestor of Judah?
if ancestor abraham judah >/dev/null 2>&1; then
    echo "✓ Yes, Abraham is an ancestor of Judah"
else
    echo "✗ No"
fi

## Schritt 11: Die Kompilierungsstrategie verstehen

Analysieren wir, was UnifyWeaver getan hat:

1. **Parent-Kompilierung**: Verwendete `stream_compiler`, um eine einfache Streaming-Funktion zu erstellen, die alle Eltern-Kind-Paare ausgibt

2. **Ancestor-Kompilierung**: Erkannte das Muster der transitiven Hülle und wendete die BFS-Optimierung (Breitensuche) an, um alle erreichbaren Vorfahren effizient zu berechnen

Überprüfen wir die Kompilierungsstrategie:

In [ ]:
% Check if ancestor is classified as recursive
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),
    recursive_compiler:classify_predicate(ancestor/2, _Classification),
    format('Ancestor classification: ~w~n', [_Classification])
).

## Zusammenfassung

In diesem Notebook haben Sie gelernt:

✅ Wie man Prolog-Fakten und -Regeln definiert

✅ Wie man `stream_compiler` von UnifyWeaver für Fakten verwendet

✅ Wie man `recursive_compiler` von UnifyWeaver für rekursive Prädikate verwendet

✅ Wie man generierte Bash-Skripte testet

✅ Dass UnifyWeaver transitive Hüllen automatisch erkennt und BFS-Optimierung anwendet

## Nächste Schritte

Probieren Sie diese Übungen aus:

1. Fügen Sie dem Stammbaum weitere Familienmitglieder hinzu
2. Definieren Sie ein `grandparent/2`-Prädikat und kompilieren Sie es
3. Erstellen Sie ein `sibling/2`-Prädikat (zwei Personen mit denselben Eltern)
4. Untersuchen Sie den generierten Bash-Code, um den BFS-Algorithmus zu verstehen

Fahren Sie mit **Notebook 2: Vergleich von Rekursionsmustern** fort, um mehr über fortgeschrittene Rekursionsmuster zu erfahren!